# Claim Extraction

This notebook extracts environmental claims from sustainability reports using the `climatebert/environmental-claims` model.

## Setup

Install the required libraries if they are not available in your environment.

In [ ]:
%pip install -q transformers accelerate torch pdfplumber pandas tqdm

## Imports

In [ ]:
import json
from pathlib import Path
from typing import Dict, List

import pandas as pd
import pdfplumber
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

## Configuration

Adjust the paths or thresholds below if needed.

In [ ]:
ESG_REPORT_DIR = Path('ESG Reports')
OUTPUT_CSV = Path('claims_extracted.csv')
MODEL_NAME = 'climatebert/environmental-claims'
MAX_TOKENS = 350
STRIDE = 50
MIN_SCORE = 0.6

## Helper Functions

In [ ]:
def iter_pdf_text(pdf_path: Path) -> List[str]:
    """Extract cleaned text chunks from a PDF file."""
    paragraphs: List[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=1, y_tolerance=1) or ''
            segments = [seg.strip() for seg in text.split('\n') if seg.strip()]
            paragraphs.extend(segments)
    return paragraphs


def chunk_text(paragraphs: List[str], max_tokens: int, stride: int, tokenizer) -> List[Dict[str, str]]:
    """Chunk text into overlapping windows for the transformer model."""
    encoded = tokenizer(paragraphs, padding=False, truncation=False, add_special_tokens=False)
    chunks: List[Dict[str, str]] = []
    current_tokens: List[int] = []
    current_texts: List[str] = []
    for text, token_ids in zip(paragraphs, encoded['input_ids']):
        if len(current_tokens) + len(token_ids) > max_tokens:
            if current_tokens:
                chunks.append({'text': ' '.join(current_texts)})
                if stride > 0:
                    stride_tokens = current_tokens[-stride:]
                    stride_text = tokenizer.decode(stride_tokens, skip_special_tokens=True)
                    current_tokens = list(stride_tokens)
                    current_texts = [stride_text]
                else:
                    current_tokens = []
                    current_texts = []
        current_tokens.extend(token_ids)
        current_texts.append(text)
    if current_tokens:
        chunks.append({'text': ' '.join(current_texts)})
    return chunks

## Load Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
claim_classifier = pipeline('text-classification', model=model, tokenizer=tokenizer, top_k=None, truncation=True)

## Process Reports

In [ ]:
records = []
for pdf_path in sorted(ESG_REPORT_DIR.glob('*.pdf')):
    paragraphs = iter_pdf_text(pdf_path)
    chunks = chunk_text(paragraphs, max_tokens=MAX_TOKENS, stride=STRIDE, tokenizer=tokenizer)
    for idx, chunk in enumerate(tqdm(chunks, desc=f'Processing {pdf_path.name}')):
        text = chunk['text']
        results = claim_classifier(text, truncation=True)
        for result in results:
            label = result['label']
            score = float(result['score'])
            if label.lower() == 'claim' and score >= MIN_SCORE:
                records.append({
                    'report': pdf_path.name,
                    'chunk_id': idx,
                    'text': text,
                    'score': score,
                })

claims_df = pd.DataFrame(records)
claims_df.to_csv(OUTPUT_CSV, index=False)
claims_df.head()

## Save and Inspect

The full set of extracted claims is stored in `claims_extracted.csv`.